# E-02: Random forest and XGBoost

Predict the historical arraignment bail/remand decision using the unchanged three nested feature sets. Tune the agreed full grids by five-fold stratified training-year ROC-AUC using the shared `fit_and_predict` function. Report training OOF and independent validation-year performance. OOF uses training-selected parameters and is not nested CV. RF uses 300 trees. Each XGBoost fit selects its tree count with an internal stratified 20% training-fold holdout (log-loss, patience 50, ceiling 2000), then refits all rows supplied to that fit. No training-dataset reduction for CV tuning, class weighting, or test-year access. XGBoost uses its configured internal per-tree row subsampling (subsample=0.8).

All model files and case-level predictions stay local. The optional `E02_RESUME=1` environment setting reuses only checkpoints matching the current data, code, package versions, and model specifications.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
from src import config
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import os
import platform
import subprocess
import time
import joblib
import numpy as np
import pandas as pd
from src import features
from src.evaluate import metrics
from src.models import PREDICTION_COLUMNS, fit_and_predict, split_by_year
from src.models_ensemble import ENSEMBLE_MODELS
from src.sample import build_sample, load_raw

assert config.ROOT == Path.cwd().resolve().parent
versions = {p: version(p) for p in ["pandas", "numpy", "scikit-learn", "xgboost", "pyarrow"]}
print("Python:", platform.python_version(), "Packages:", versions)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=config.ROOT, text=True).strip())


Python: 3.11.6 Packages: {'pandas': '3.0.1', 'numpy': '2.4.3', 'scikit-learn': '1.8.0', 'xgboost': '3.2.0', 'pyarrow': '26.0.1'}
Source commit: 81c4231df95a91e0c8440fd53171f4e98e2eb96c


## Synthetic early-stopping safeguards

These small synthetic checks run before reading the development data. They check the estimator boundary, sklearn cloning, serialization, and exclusion of development validation/test outcomes from fitting.

In [2]:
from sklearn.base import clone, is_classifier
from src.models_ensemble import TrainOnlyEarlyStoppingXGBClassifier
from xgboost import XGBClassifier
import runpy
import tempfile

# Noisy synthetic labels make stopping before the ceiling observable.
rng = np.random.default_rng(config.RANDOM_STATE)
noise_x = rng.normal(size=(500, 5))
noise_y = rng.integers(0, 2, 500)
small = clone(ENSEMBLE_MODELS["xgb"][0]).set_params(
    n_estimators=100, early_stopping_rounds=5, n_jobs=1,
    max_depth=3, learning_rate=0.1)
calls = []
original_fit = XGBClassifier.fit
def traced_fit(estimator, x, y, **kwargs):
    evaluation = kwargs.get("eval_set") or []
    calls.append((len(y), sum(len(labels) for _, labels in evaluation)))
    return original_fit(estimator, x, y, **kwargs)
try:
    XGBClassifier.fit = traced_fit
    small.fit(noise_x, noise_y)
finally:
    XGBClassifier.fit = original_fit
assert calls == [(400, 100), (500, 0)]
assert small.stopped_early_ and 1 <= small.selected_n_estimators_ < 100
assert is_classifier(small) and clone(small).n_estimators == 100
assert small.get_booster().num_boosted_rounds() == small.selected_n_estimators_
assert small.best_iteration + 1 == small.selected_n_estimators_
try:
    small.fit(noise_x, noise_y, eval_set=[(noise_x, noise_y)])
except ValueError:
    pass
else:
    raise AssertionError("External early-stopping data must be rejected.")

# Test the unchanged shared Pipeline with poisoned test labels and reversed
# validation labels. Neither can change training, tree counts or probabilities.
fixture = runpy.run_path(str(config.ROOT / "tests/test_models.py"))["_development_fixture"]()
original_models_dir = config.MODELS_DIR
try:
    with tempfile.TemporaryDirectory() as temporary_directory:
        config.MODELS_DIR = Path(temporary_directory)
        temporary_model = config.MODELS_DIR / "synthetic.joblib"
        joblib.dump(small, temporary_model)
        np.testing.assert_array_equal(joblib.load(temporary_model).predict_proba(noise_x), small.predict_proba(noise_x))
        estimator = clone(small).set_params(n_estimators=30, early_stopping_rounds=3)
        args = dict(model_name="xgb", estimator=estimator,
            param_grid={"clf__max_depth": [3]}, feature_set="CASE_COURT_JUDGE", scale=False)
        before = fit_and_predict(fixture, **args)
        fitted_path = config.MODELS_DIR / f"{config.TARGET_Q1}_CASE_COURT_JUDGE_xgb.joblib"
        before_rounds = joblib.load(fitted_path).named_steps["clf"].selected_n_estimators_
        fixture.loc[fixture[config.SPLIT_COL].eq("valid"), config.TARGET_Q1] ^= 1
        after = fit_and_predict(fixture, **args)
        classifier = joblib.load(fitted_path).named_steps["clf"]
        assert before.attrs["best_params"] == after.attrs["best_params"]
        np.testing.assert_array_equal(before.y_prob, after.y_prob)
        assert before_rounds == classifier.selected_n_estimators_
        assert classifier.refit_n_ == 80 and classifier.early_stopping_fit_n_ == 64
        assert classifier.early_stopping_valid_n_ == 16 and clone(classifier).n_estimators == 30
        assert set(after["split"]) == {"train_oof", "valid"} and not after.y_true.eq(9).any()
finally:
    config.MODELS_DIR = original_models_dir
print("Early-stopping checks passed: internal holdout, full-row refit, actual stopping, clone/serialization, external-eval rejection, test sentinels and validation-label invariance.")


Early-stopping checks passed: internal holdout, full-row refit, actual stopping, clone/serialization, external-eval rejection, test sentinels and validation-label invariance.


In [3]:
development_years = [*config.TRAIN_YEARS, config.VALID_YEAR]
assert config.TEST_YEAR not in development_years
columns = list(dict.fromkeys([
    config.ID_COL, config.YEAR_COL, config.SPLIT_COL, config.MAIN_SAMPLE_COL,
    config.TARGET_Q1, *[c for cols in config.FEATURE_SETS.values() for c in cols],
]))
published = pd.read_csv(config.SAMPLE_FLOW_FILE).set_index("step")
if config.ANALYTIC_FILE.is_file():
    development = pd.read_parquet(config.ANALYTIC_FILE, columns=columns,
        filters=[(config.YEAR_COL, "in", development_years)])
    source = "analytic.parquet: development-year partitions only"
else:
    raw = load_raw(years=development_years)
    assert set(raw[config.YEAR_COL].unique()) == set(development_years)
    development, flow = build_sample(raw)
    del raw
    pd.testing.assert_frame_equal(
        flow.set_index("step")[[str(y) for y in development_years]],
        published[[str(y) for y in development_years]], check_dtype=False)
    development = development[columns]
    source = "rebuilt development years; all published sample-flow steps matched"
main = development.loc[development[config.MAIN_SAMPLE_COL]].copy()
del development
assert set(main[config.YEAR_COL].unique()) == set(development_years)
assert main.groupby(config.YEAR_COL).size().to_dict() == {
    year: int(published.loc["MAIN", str(year)]) for year in development_years}
parts = split_by_year(main)
assert parts["test"].empty
for fs, cols in config.FEATURE_SETS.items():
    features.check_leakage(cols)
    assert not set(cols) & set(config.FAIRNESS_GROUPS)
print("Input:", source)
print("Main cases:", main.groupby(config.YEAR_COL).size().to_dict())
print("Feature counts:", {fs: len(cols) for fs, cols in config.FEATURE_SETS.items()})
checkpoint_dir = config.DATA_PROCESSED / "e02_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
source_hashes = {p: hashlib.sha256((config.ROOT / p).read_bytes()).hexdigest()
    for p in ["src/config.py", "src/sample.py", "src/features.py", "src/models.py", "src/models_ensemble.py"]}
data_hash = hashlib.sha256(pd.util.hash_pandas_object(main, index=True).values.tobytes()).hexdigest()
specifications = {name: {"estimator": repr(est), "grid": grid} for name, (est, grid) in ENSEMBLE_MODELS.items()}
manifest = {"source": source, "source_hashes": source_hashes, "data_sha256": data_hash,
    "versions": versions, "python": platform.python_version(), "specifications": specifications,
    "years": development_years, "counts": {str(y): int(n) for y, n in main.groupby(config.YEAR_COL).size().items()}}
def pair_fingerprint(name, feature_set):
    shared = {p: digest for p, digest in source_hashes.items() if p != "src/models_ensemble.py"}
    specification = {"data_sha256": data_hash, "versions": versions, "shared_sources": shared,
        "python": platform.python_version(), "model": name, "feature_set": feature_set,
        "specification": specifications[name]}
    if name == "xgb":
        specification["estimator_source_sha256"] = source_hashes["src/models_ensemble.py"]
    return hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()

legacy_manifest_path = checkpoint_dir / "manifest.json"
legacy_manifest = json.loads(legacy_manifest_path.read_text()) if legacy_manifest_path.exists() else None
resume = os.environ.get("E02_RESUME", "0") == "1"
predictions, diagnostics, tuning, apparent = [], [], [], []


  read NYS for Web 2021.csv.gz: 246,202 rows (124,264 NYC)


  read NYS for Web 2022.csv.gz: 286,588 rows (150,585 NYC)


  read NYS for Web 2023.csv.gz: 324,619 rows (175,658 NYC)


Input: rebuilt development years; all published sample-flow steps matched
Main cases: {np.int64(2021): 89189, np.int64(2022): 104007, np.int64(2023): 120457}
Feature counts: {'CASE': 18, 'CASE_COURT': 19, 'CASE_COURT_JUDGE': 20}


In [4]:
def fit_pair(name, feature_set):
    """Call the shared workflow and checkpoint provenance per model specification."""
    key = f"{name}_{feature_set}"
    fingerprint = pair_fingerprint(name, feature_set)
    part_path = checkpoint_dir / f"{key}.parquet"
    meta_path = checkpoint_dir / f"{key}.json"
    model_path = config.MODELS_DIR / f"{config.TARGET_Q1}_{feature_set}_{name}.joblib"
    print("Processing", key, flush=True)
    meta = None
    if resume and part_path.is_file() and meta_path.is_file() and model_path.is_file():
        candidate = json.loads(meta_path.read_text())
        valid = candidate.get("checkpoint_version") == 2 and candidate["fingerprint"] == fingerprint
        # RF is unchanged. Migrate its original checkpoint only after verifying
        # identical data, shared code, library versions and stock RF/grid repr.
        if not valid and name == "rf" and legacy_manifest is not None:
            old = legacy_manifest
            legacy_hash = hashlib.sha256(json.dumps(old, sort_keys=True).encode()).hexdigest()
            valid = candidate["fingerprint"] == legacy_hash
            valid &= all(old[k] == manifest[k] for k in ["data_sha256", "versions", "python", "years", "counts"])
            valid &= old["specifications"]["rf"] == specifications["rf"]
            valid &= all(old["source_hashes"][p] == digest for p, digest in source_hashes.items()
                         if p != "src/models_ensemble.py")
        if valid:
            assert candidate["fit_diagnostics"]["model"] == name
            assert candidate["fit_diagnostics"]["feature_set"] == feature_set
            assert candidate["model_sha256"] == hashlib.sha256(model_path.read_bytes()).hexdigest()
            assert candidate["prediction_sha256"] == hashlib.sha256(part_path.read_bytes()).hexdigest()
            meta = candidate
            meta.update(fingerprint=fingerprint, checkpoint_version=2)
            meta_path.write_text(json.dumps(meta, indent=2))
            out = pd.read_parquet(part_path)
            print("Reused verified checkpoint", key)
    if meta is None:
        estimator, grid = ENSEMBLE_MODELS[name]
        out = fit_and_predict(main, name, estimator, grid, feature_set,
            target=config.TARGET_Q1, run_test=False, scale=False)
        fitted = joblib.load(model_path)
        train_p = fitted.predict_proba(features.get_X(parts["train"], feature_set))[:, 1]
        meta = {"fingerprint": fingerprint, "checkpoint_version": 2,
            "fit_diagnostics": out.attrs["fit_diagnostics"],
            "tuning_results": out.attrs["tuning_results"],
            "apparent": {"model": name, "feature_set": feature_set, "split": "train_apparent",
                **metrics(parts["train"][config.TARGET_Q1], train_p)}}
        if name == "xgb":
            classifier = fitted.named_steps["clf"]
            meta["early_stopping"] = {"model": name, "feature_set": feature_set,
                "ceiling": classifier.n_estimators, "patience": classifier.early_stopping_rounds,
                **{attr: getattr(classifier, attr) for attr in ["selected_n_estimators_",
                    "early_stopping_best_score_", "early_stopping_rounds_trained_",
                    "early_stopping_fit_n_", "early_stopping_valid_n_", "refit_n_", "stopped_early_"]}}
        out.attrs = {}
        out.to_parquet(part_path, index=False)
        meta["model_sha256"] = hashlib.sha256(model_path.read_bytes()).hexdigest()
        meta["prediction_sha256"] = hashlib.sha256(part_path.read_bytes()).hexdigest()
        meta_path.write_text(json.dumps(meta, indent=2))
    assert out.columns.tolist() == PREDICTION_COLUMNS
    assert set(out["split"]) == {"train_oof", "valid"}
    assert set(out["model"]) == {name} and set(out["feature_set"]) == {feature_set}
    assert not out[config.YEAR_COL].eq(config.TEST_YEAR).any()
    for split, ref in [("train_oof", parts["train"]), ("valid", parts["valid"])]:
        got = out.loc[out["split"].eq(split)].set_index(config.ID_COL).sort_index()
        expected = ref.set_index(config.ID_COL).sort_index()
        assert got.index.equals(expected.index)
        np.testing.assert_array_equal(got["y_true"], expected[config.TARGET_Q1])
        print(split, metrics(got["y_true"], got["y_prob"]))
    diagnostics.append(meta["fit_diagnostics"])
    tuning.extend(meta["tuning_results"])
    apparent.append(meta["apparent"])
    predictions.append(out)
    print("Selected:", meta["fit_diagnostics"]["best_params"], flush=True)
    if name == "xgb":
        print("Training-only early stopping:", meta["early_stopping"])


### RF / CASE

In [5]:
fit_pair("rf", "CASE")

Processing rf_CASE


Reused verified checkpoint rf_CASE


train_oof {'roc_auc': 0.940662738068091, 'avg_precision': 0.7297894191321784, 'brier': 0.05987922595819726, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9253558478886296, 'avg_precision': 0.6686184749682536, 'brier': 0.0667246316980198, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__max_depth": null, "clf__min_samples_leaf": 20}


### RF / CASE_COURT

In [6]:
fit_pair("rf", "CASE_COURT")

Processing rf_CASE_COURT


Reused verified checkpoint rf_CASE_COURT


train_oof {'roc_auc': 0.941951317219854, 'avg_precision': 0.7355932475797411, 'brier': 0.05929789796840796, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9258490563407918, 'avg_precision': 0.6733215550323558, 'brier': 0.06632309797466215, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__max_depth": null, "clf__min_samples_leaf": 20}


### RF / CASE_COURT_JUDGE

In [7]:
fit_pair("rf", "CASE_COURT_JUDGE")

Processing rf_CASE_COURT_JUDGE


Reused verified checkpoint rf_CASE_COURT_JUDGE


train_oof {'roc_auc': 0.9403081737806396, 'avg_precision': 0.7336260736052858, 'brier': 0.0603714717493674, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9235175864198844, 'avg_precision': 0.6680588157767299, 'brier': 0.06724639037385102, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__max_depth": null, "clf__min_samples_leaf": 20}


### XGB / CASE

In [8]:
fit_pair("xgb", "CASE")

Processing xgb_CASE


Reused verified checkpoint xgb_CASE


train_oof {'roc_auc': 0.9431228887359695, 'avg_precision': 0.7378706224358214, 'brier': 0.0584605087797356, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9303046836458405, 'avg_precision': 0.6832361055206122, 'brier': 0.0650621390303114, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__learning_rate": 0.05, "clf__max_depth": 5, "clf__subsample": 0.8}


Training-only early stopping: {'model': 'xgb', 'feature_set': 'CASE', 'ceiling': 2000, 'patience': 50, 'selected_n_estimators_': 470, 'early_stopping_best_score_': 0.18957954529135207, 'early_stopping_rounds_trained_': 520, 'early_stopping_fit_n_': 154556, 'early_stopping_valid_n_': 38640, 'refit_n_': 193196, 'stopped_early_': True}


### XGB / CASE_COURT

In [9]:
fit_pair("xgb", "CASE_COURT")

Processing xgb_CASE_COURT


Reused verified checkpoint xgb_CASE_COURT


train_oof {'roc_auc': 0.9451722586607865, 'avg_precision': 0.747703006047705, 'brier': 0.057382957337000995, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9312175206792328, 'avg_precision': 0.6903056324717098, 'brier': 0.0644257633918234, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__learning_rate": 0.05, "clf__max_depth": 5, "clf__subsample": 0.8}


Training-only early stopping: {'model': 'xgb', 'feature_set': 'CASE_COURT', 'ceiling': 2000, 'patience': 50, 'selected_n_estimators_': 651, 'early_stopping_best_score_': 0.1871267316444669, 'early_stopping_rounds_trained_': 701, 'early_stopping_fit_n_': 154556, 'early_stopping_valid_n_': 38640, 'refit_n_': 193196, 'stopped_early_': True}


### XGB / CASE_COURT_JUDGE

In [10]:
fit_pair("xgb", "CASE_COURT_JUDGE")

Processing xgb_CASE_COURT_JUDGE


Reused verified checkpoint xgb_CASE_COURT_JUDGE


train_oof {'roc_auc': 0.949179558997103, 'avg_precision': 0.7667366014707199, 'brier': 0.05508142764635243, 'n': 193196, 'base_rate': 0.12929356715459947}


valid {'roc_auc': 0.9346907259303342, 'avg_precision': 0.7035674570797144, 'brier': 0.06301980084744181, 'n': 120457, 'base_rate': 0.1281785201358161}
Selected: {"clf__learning_rate": 0.05, "clf__max_depth": 7, "clf__subsample": 0.8}


Training-only early stopping: {'model': 'xgb', 'feature_set': 'CASE_COURT_JUDGE', 'ceiling': 2000, 'patience': 50, 'selected_n_estimators_': 878, 'early_stopping_best_score_': 0.181059400972203, 'early_stopping_rounds_trained_': 928, 'early_stopping_fit_n_': 154556, 'early_stopping_valid_n_': 38640, 'refit_n_': 193196, 'stopped_early_': True}


## Aggregate performance and checks

Training OOF measures held-out-fold ranking using training-selected parameters; validation assesses transfer to the next year. Apparent training performance is reported separately to reveal optimism from scoring fitted cases. Final model selection and paired uncertainty intervals belong to BD-06.

In [11]:
pred_df = pd.concat(predictions, ignore_index=True)[PREDICTION_COLUMNS]
keys = [config.ID_COL, "model", "feature_set", "target", "split"]
assert not pred_df.duplicated(keys).any()
assert len(pred_df) == len(ENSEMBLE_MODELS) * len(config.FEATURE_SETS) * len(main)
assert np.isfinite(pred_df["y_prob"]).all() and pred_df["y_prob"].between(0, 1).all()
temporary = config.PRED_ENSEMBLE_FILE.with_suffix(".pending.parquet")
pred_df.to_parquet(temporary, index=False)
temporary.replace(config.PRED_ENSEMBLE_FILE)
summary = pd.DataFrame([
    {"model": name, "feature_set": fs, "split": split, **metrics(g.y_true, g.y_prob)}
    for (name, fs, split), g in pred_df.groupby(["model", "feature_set", "split"], sort=False)])
summary.to_csv(config.REPORTS / "e02_metrics.csv", index=False)
pd.DataFrame(diagnostics).to_csv(config.REPORTS / "e02_fit_diagnostics.csv", index=False)
pd.DataFrame(tuning).to_csv(config.REPORTS / "e02_tuning.csv", index=False)
pd.DataFrame(apparent).to_csv(config.REPORTS / "e02_train_apparent.csv", index=False)
(checkpoint_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
auc = summary.pivot(index=["model", "feature_set"], columns="split", values="roc_auc")
auc["oof_minus_valid"] = auc["train_oof"] - auc["valid"]
print(auc.to_string(float_format=lambda x: f"{x:.6f}"))
baseline = pd.read_csv(config.REPORTS / "bd05_metrics.csv")
combined = pd.concat([baseline, summary], ignore_index=True)
print("Validation comparison with published baselines:")
print(combined.loc[combined.split.eq("valid") & combined.feature_set.isin(config.FEATURE_SETS)]
    .pivot(index="model", columns="feature_set", values="roc_auc").to_string(float_format=lambda x: f"{x:.6f}"))
print("Apparent training diagnostics (not OOF):")
print(pd.DataFrame(apparent).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

early_stopping_table = pd.DataFrame([
    json.loads((checkpoint_dir / f"xgb_{fs}.json").read_text())["early_stopping"]
    for fs in config.FEATURE_SETS])
assert early_stopping_table.refit_n_.eq(len(parts["train"])).all()
assert (early_stopping_table.early_stopping_fit_n_ + early_stopping_table.early_stopping_valid_n_).eq(len(parts["train"])).all()
early_stopping_table.to_csv(config.REPORTS / "e02_early_stopping.csv", index=False)
print(early_stopping_table.to_string(index=False))


model      feature_set     split  roc_auc  avg_precision    brier      n  base_rate
   rf             CASE train_oof 0.940663       0.729789 0.059879 193196   0.129294
   rf             CASE     valid 0.925356       0.668618 0.066725 120457   0.128179
   rf       CASE_COURT train_oof 0.941951       0.735593 0.059298 193196   0.129294
   rf       CASE_COURT     valid 0.925849       0.673322 0.066323 120457   0.128179
   rf CASE_COURT_JUDGE train_oof 0.940308       0.733626 0.060371 193196   0.129294
   rf CASE_COURT_JUDGE     valid 0.923518       0.668059 0.067246 120457   0.128179
  xgb             CASE train_oof 0.943123       0.737871 0.058461 193196   0.129294
  xgb             CASE     valid 0.930305       0.683236 0.065062 120457   0.128179
  xgb       CASE_COURT train_oof 0.945172       0.747703 0.057383 193196   0.129294
  xgb       CASE_COURT     valid 0.931218       0.690306 0.064426 120457   0.128179
  xgb CASE_COURT_JUDGE train_oof 0.949180       0.766737 0.055081 193196   0

The saved predictions contain development years only. Inspect both AUC and probability metrics; extra court/judge inputs may help some algorithms more than others. These are predictions of bail/remand decisions and do not measure actual detention, future offending, or causal judge effects.

## Saved-model verification and report

In [12]:
# Reproduce every validation probability from the saved model.
for name in ENSEMBLE_MODELS:
    for fs in config.FEATURE_SETS:
        fitted = joblib.load(config.MODELS_DIR / f"{config.TARGET_Q1}_{fs}_{name}.joblib")
        x_train = features.get_X(parts["train"], fs)
        x_valid = features.get_X(parts["valid"], fs)
        preprocess = fitted.named_steps["preprocess"]
        categorical = [c for c in config.FEATURE_SETS[fs] if c in features.CATEGORICAL]
        for column, categories in zip(categorical, preprocess.named_transformers_["categorical"].categories_):
            assert set(categories) == set(x_train[column].unique())
        if name == "xgb":
            from sklearn.base import clone
            classifier = fitted.named_steps["clf"]
            assert classifier.refit_n_ == len(x_train)
            assert clone(classifier).n_estimators == classifier.n_estimators == 2000
            assert classifier.get_booster().num_boosted_rounds() == classifier.selected_n_estimators_
        probability = fitted.predict_proba(x_valid)[:, 1]
        saved = pred_df.loc[pred_df.model.eq(name) & pred_df.feature_set.eq(fs) & pred_df.split.eq("valid")]
        saved = saved.set_index(config.ID_COL).loc[parts["valid"][config.ID_COL]]
        np.testing.assert_allclose(probability, saved.y_prob.to_numpy(), atol=1e-12, rtol=0)
        assert not any(c in config.FAIRNESS_GROUPS or c in config.LEAKAGE_BLOCKLIST for c in fitted.feature_names_in_)
        print(name, fs, "saved-model probabilities reproduced; encoder categories are training-only")
pd.testing.assert_frame_equal(pd.read_parquet(config.PRED_ENSEMBLE_FILE), pred_df)
assert not pred_df[config.YEAR_COL].eq(config.TEST_YEAR).any()
print("All six fits verified. Case-level predictions and models remain local.")


rf CASE saved-model probabilities reproduced; encoder categories are training-only


rf CASE_COURT saved-model probabilities reproduced; encoder categories are training-only


rf CASE_COURT_JUDGE saved-model probabilities reproduced; encoder categories are training-only


xgb CASE saved-model probabilities reproduced; encoder categories are training-only


xgb CASE_COURT saved-model probabilities reproduced; encoder categories are training-only


xgb CASE_COURT_JUDGE saved-model probabilities reproduced; encoder categories are training-only


All six fits verified. Case-level predictions and models remain local.


In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def markdown_table(frame, digits=6):
    formatted = frame.copy()
    for column in formatted:
        if pd.api.types.is_float_dtype(formatted[column]):
            formatted[column] = formatted[column].map(lambda value: "n/a" if pd.isna(value) else f"{value:.{digits}f}")
    rows = ["| " + " | ".join(map(str, formatted.columns)) + " |",
            "| " + " | ".join(["---"] * len(formatted.columns)) + " |"]
    rows.extend("| " + " | ".join(map(str, row)) + " |"
                for row in formatted.itertuples(index=False, name=None))
    return "\n".join(rows)

sets = list(config.FEATURE_SETS)
auc_table = summary.pivot(index=["model", "feature_set"], columns="split", values="roc_auc")
apparent_table = pd.DataFrame(apparent).set_index(["model", "feature_set"])
auc_table["train_apparent"] = apparent_table["roc_auc"]
auc_table = auc_table[["train_apparent", "train_oof", "valid"]].reset_index()
increments = []
valid_scores = summary.loc[summary.split.eq("valid")].set_index(["model", "feature_set"])
for name in ENSEMBLE_MODELS:
    for before, after in zip(sets, sets[1:]):
        increments.append({"model": name, "comparison": f"{after} minus {before}",
            **{f"delta_{metric}": float(valid_scores.loc[(name, after), metric] - valid_scores.loc[(name, before), metric])
               for metric in ["roc_auc", "avg_precision", "brier"]}})
increments = pd.DataFrame(increments)
increments.to_csv(config.REPORTS / "e02_nested_gains_valid.csv", index=False)

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.7), sharey=True)
low = np.floor((summary.roc_auc.min() - 0.004) * 100) / 100
high = np.ceil((summary.roc_auc.max() + 0.004) * 100) / 100
for ax, name in zip(axes, ENSEMBLE_MODELS):
    for split, label, color, marker in [("train_oof", "Training OOF", "#65768A", "o"),
                                       ("valid", "Validation", "#007E87", "s")]:
        values = [float(summary.loc[summary.model.eq(name) & summary.feature_set.eq(fs) & summary.split.eq(split), "roc_auc"].iloc[0]) for fs in sets]
        ax.plot(range(len(sets)), values, marker=marker, color=color, label=label, linewidth=2)
        for i, value in enumerate(values):
            ax.annotate(f"{value:.4f}", (i, value), xytext=(0, 8 if split == "train_oof" else -17), textcoords="offset points", ha="center", fontsize=10)
    ax.set(title={"rf": "Random forest", "xgb": "XGBoost (early stopping)"}[name], xticks=range(len(sets)),
           xticklabels=["Case", "+ County", "+ Judge"], ylim=(low, high), xlim=(-0.25, 2.25))
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("ROC-AUC")
axes[1].legend(loc="lower left", frameon=False)
fig.suptitle("Ensemble performance across nested feature sets", fontsize=15)
fig.text(0.08, 0.025, f"Source: NY UCS Pretrial Release Data; NYC. Train: {config.TRAIN_YEARS}; validation: {config.VALID_YEAR}.\nOOF uses training-selected parameters; performance shown on a truncated AUC axis.", fontsize=9, color="#444444")
fig.tight_layout(rect=(0, 0.1, 1, 0.95))
config.FIGURES.mkdir(exist_ok=True, parents=True)
fig.savefig(config.FIGURES / "e02_auc_comparison.png", dpi=180)
plt.close(fig)

comparison = combined.loc[combined.split.eq("valid") & combined.feature_set.isin(config.FEATURE_SETS)]
comparison = comparison.pivot(index="model", columns="feature_set", values="roc_auc").reindex(columns=sets).reset_index()
winner = summary.loc[summary.split.eq("valid")].sort_values("roc_auc", ascending=False).iloc[0]
baseline_best = baseline.loc[baseline.split.eq("valid") & baseline.feature_set.isin(sets)].sort_values("roc_auc", ascending=False).iloc[0]
training_n, valid_n = len(parts["train"]), len(parts["valid"])
elapsed = sum(row["elapsed_seconds"] for row in diagnostics)
early_display = early_stopping_table[["feature_set", "selected_n_estimators_",
    "early_stopping_rounds_trained_", "early_stopping_best_score_",
    "early_stopping_valid_n_", "refit_n_"]].rename(columns={
        "selected_n_estimators_": "selected_trees", "early_stopping_rounds_trained_": "rounds_evaluated",
        "early_stopping_best_score_": "best_internal_log_loss", "early_stopping_valid_n_": "internal_holdout_n",
        "refit_n_": "refit_n"})
report = f'''# E-02 random forest and XGBoost: development-year results

## Main result

The highest validation AUC among the six ensemble fits is **{winner["model"]} / {winner["feature_set"]}: {winner["roc_auc"]:.6f}**. The best published main baseline is {baseline_best["model"]} / {baseline_best["feature_set"]}, AUC {baseline_best["roc_auc"]:.6f}; the point difference is {winner["roc_auc"] - baseline_best["roc_auc"]:+.6f}. This comparison does not select the final model; BD-06 must consider paired uncertainty, calibration, and fairness.

RF performs best with CASE_COURT (validation AUC {valid_scores.loc[("rf", "CASE_COURT"), "roc_auc"]:.6f}); adding judge changes its AUC by {valid_scores.loc[("rf", "CASE_COURT_JUDGE"), "roc_auc"] - valid_scores.loc[("rf", "CASE_COURT"), "roc_auc"]:.6f}. For XGBoost, adding county changes AUC by {valid_scores.loc[("xgb", "CASE_COURT"), "roc_auc"] - valid_scores.loc[("xgb", "CASE"), "roc_auc"]:.6f}, and adding judge changes it by {valid_scores.loc[("xgb", "CASE_COURT_JUDGE"), "roc_auc"] - valid_scores.loc[("xgb", "CASE_COURT"), "roc_auc"]:.6f}. County adds little in both ensembles; judge gains depend on the algorithm and representation.

All six OOF AUCs exceed their future-year validation AUCs. Apparent training AUC reaches {apparent_table.roc_auc.max():.6f}, showing why scoring fitted cases should be separated from OOF and validation performance. The primary interpretation uses validation, with the training/validation gap reported below.

## Training and validation AUC

{markdown_table(auc_table)}

- `train_apparent`: fitted full-training model scored on the same training cases; optimistic and supplementary.
- `train_oof`: five-fold out-of-fold predictions on the training years, using training-selected hyperparameters. Preprocessing and model fitting occur within each fold. This is not nested cross-validation because tuning used all training folds.
- `valid`: the full-training fitted model scored on the independent validation year. Use this column for the primary comparison.

![Training OOF and validation AUC](figures/e02_auc_comparison.png)

## Added county and judge information

{markdown_table(increments)}

Positive AUC/AP increments and negative Brier increments favor the larger feature set. These are point estimates on identical validation cases. Judge information may behave differently across algorithms. A small or negative gain is evidence against assuming that extra identifiers always improve future-year prediction. Gains are predictive associations; assignment and unmeasured case mix remain possible explanations. The RF decline is specific to the current estimator and grid: its default max_features="sqrt" changes the number and proportion of split candidates as one-hot dimensions grow, so the decline does not establish that judge information has no signal. BD-06 should additionally report cases with individually coded training-year judges because `unseen` cases cannot use an individual judge coefficient or split.

## Comparison with published baselines

{markdown_table(comparison)}

The charge rule uses only charge weight and the violent-felony flag; its CASE label does not mean it uses all CASE inputs. AUC is a ranking measure and is not classification accuracy. AP and Brier provide complementary information.

## All development metrics

{markdown_table(summary)}

## Training-only early stopping

{markdown_table(early_display)}

Each GridSearchCV fit, selected-model full-training fit and OOF fit repeats an internal 80/20 stratified split of only the rows supplied to that fit. Log-loss chooses the best boosting iteration with patience 50 and ceiling 2000; a fresh booster then refits every supplied row for that many rounds. The ceiling is preserved by sklearn clone, so OOF fits choose their own counts. The shared preprocessor is fitted on the outer training fold, including the internal early-stopping slice; it sees neither outer CV holdouts nor the validation year. Internal early-stopping log-loss is a tuning diagnostic, not an independent performance estimate.

The earlier fixed-400 run was exploratory and is superseded. Early stopping is applied identically across candidates without choosing the policy based on validation-year scores. The final-test workflow should freeze this policy with the chosen hyperparameters before refitting the expanded training years; it must never early-stop on the test year.

## Specifications and provenance

- Source checkout HEAD at execution: `{subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=config.ROOT, text=True).strip()}`. Only E-owned ensemble code and notebook are added; shared functions and feature definitions are unchanged.
- Training years: {config.TRAIN_YEARS}, n={training_n:,}; validation year: {config.VALID_YEAR}, n={valid_n:,}. All development-year sample-flow steps matched the published counts when rebuilding from raw data. Test-year raw data, outcomes and predictions were not used.
- Three original nested sets: CASE (18 inputs), CASE_COURT (19), CASE_COURT_JUDGE (20). Race, ethnicity and gender are absent from the main models. No data-driven reduction from 18 to 14 CASE inputs was applied.
- Shared `fit_and_predict`, five-fold stratified shuffled CV, seed from `config.RANDOM_STATE` ({config.RANDOM_STATE}), scoring ROC-AUC. All six grid combinations per model are evaluated for each set on the full training sample. No reduction of the training dataset for tuning, class weighting or class oversampling. XGBoost retains its configured internal row subsampling (subsample=0.8).
- RF: 300 trees; max_depth in [8, 16, None]; min_samples_leaf in [20, 100]. XGBoost: ceiling 2000 trees, training-only log-loss early stopping with patience 50, histogram method, CPU with 8 threads; max_depth in [3, 5, 7], learning_rate in [0.05, 0.1], subsample=0.8. RF uses a fixed tree count; XGBoost selects its tree count within each training fit, then refits all supplied rows. No validation-year early stopping.
- Training-fitted one-hot encoding, unknown categories ignored, numeric gaps computationally zero-filled with `dcjs_missing` retained, no numeric scaling. Residual missing age follows the unchanged shared imputation specification. DCJS missingness is not interpreted as no prior record.
- Python {platform.python_version()}; packages: {json.dumps(versions, sort_keys=True)}. Summed shared-fit time for the six combinations: {elapsed / 60:.1f} minutes, excluding source rebuild, verification and reporting.

## Review artifacts and local files

- [Metrics](e02_metrics.csv), [fit diagnostics and selected parameters](e02_fit_diagnostics.csv), [all tuning candidates](e02_tuning.csv), [apparent training diagnostics](e02_train_apparent.csv), [nested validation gains](e02_nested_gains_valid.csv), [early-stopping diagnostics](e02_early_stopping.csv).
- Reproducible workflow: `notebooks/06_models_ensemble.ipynb`; definitions: `src/models_ensemble.py`.
- Local only: `data/processed/predictions_ensemble.parquet`, six `models/y_detain_*_rf.joblib` / `*_xgb.joblib` files and provenance-checked checkpoints. Case-level predictions and fitted models must not be committed.

Verification: the shared workflow, metric and feature tests passed (34 tests); both ensemble estimators passed a synthetic shared-pipeline smoke check with test-label sentinels; early-stopping checks confirmed actual stopping on noisy data, cloning/serialization and full-row refits, rejected external eval_set, and unchanged predictions/tree counts after validation-label reversal. Every saved model reproduced its validation probabilities to 1e-12; training-only encoder categories, prediction keys, coverage and labels were checked. The three raw development files matched the SHA-256 prefixes published in data/README.md.

Prior/pending-record reference dates remain undocumented (data/README.md D14). The outcome is a historical bail/remand decision; it does not measure actual detention or future offending. These results do not establish causal effects or discriminatory treatment.
'''
(config.REPORTS / "e02_ensemble_report.md").write_text(report, encoding="utf-8")
print("Report:", config.REPORTS / "e02_ensemble_report.md")
print("Nested validation changes:")
print(increments.to_string(index=False, float_format=lambda value: f"{value:+.6f}"))


Report: D:\wyq\Documents\ny-pretrial-detention-main\git-checkout\reports\e02_ensemble_report.md
Nested validation changes:
model                        comparison  delta_roc_auc  delta_avg_precision  delta_brier
   rf             CASE_COURT minus CASE      +0.000493            +0.004703    -0.000402
   rf CASE_COURT_JUDGE minus CASE_COURT      -0.002331            -0.005263    +0.000923
  xgb             CASE_COURT minus CASE      +0.000913            +0.007070    -0.000636
  xgb CASE_COURT_JUDGE minus CASE_COURT      +0.003473            +0.013262    -0.001406
